In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt

from interflib.io.images import load_image
from interflib.fringe.profile_fit import FringeFractionMeasurement
from interflib.dimensional.mefm import MonochromaticEFMSolver
from interflib.dimensional.refraction import EdlenRefractiveIndex

In [ ]:
# Input data

# Interferogram loading



# Wavelenght
vacuum_wavelength = 633 # nm
vacuum_wavelength_um = vacuum_wavelength / 1000 # [um]


# Gauge block information known in advance 
nominal_length_ = 100.0          # [mm]
delta_nominal_length_ = -60.0    # Tomado de certificado
uncertainty_ = 90.0              # (0.03 + (6e-4)*100)*1e3 # nm | Tomado de certificado

# Environmental conditions
air_temperature = 20             # °C
atmospheric_pressure = 75000     # Pa
air_humidity = 50                # %







In [ ]:
# Correción indice de refración y longitud de onda en aire
env = EdlenRefractiveIndex(temperature=air_temperature, pressure=atmospheric_pressure, relative_humidity=air_humidity)
index_refraction = env.index_refraction(vacuum_wavelength_um)
wavelength_air = env.wavelength_in_air(vacuum_wavelength_um) # [um]

print(f"Refractive index of air: {index_refraction:.9f}")
print(f"Wavelength in air: {wavelength_air*1e3:.6f} nm")

In [ ]:
# Fringe fraction computation
img_label = "block_100.0001582_mm(simulated).png"
img = load_image(f"../simulations/interferograms/{img_label}")

roi_platen = (700, 1900, 500, 900)   # (y1, y2, x1, x2)
roi_block = (700, 1900, 1000, 1400)

measurement = FringeFractionMeasurement(img, roi_platen, roi_block)
result = measurement.measure()
fraction = result.fringe_fraction    # <--- desired fringe fraction
print(f"Measured fringe fraction: {fraction:.4f}")
print(f"Fit R^2: platen {result.platen_quality.r2:.4f}  block {result.block_quality.r2:.4f}")

In [ ]:
# Excess fraction computation ("real" length with no corrections)
he_ne_laser = MonochromaticEFMSolver(wavelength=wavelength_air * 1e3) # entra lambda en nm
# he_ne_laser = MonochromaticEFMSolver(wavelength=632.87)

# Evaluate the 100 mm block with the mechanical and optical data
try:
    result = he_ne_laser.evaluate_block(
        nominal_length=100e6,           
        delta_nominal_length=-60.0,  # Tomado de certificado
        uncertainty=90.0,            # (0.03 + (6e-4)*100)*1e3 # nm | Tomado de certificado  
        measured_fraction=0.3908     # valor obtenido por el script para bloque de 100.0 mm
    )
        
    print("Calibration Results:")
    print("-" * 20)
    print(f"Input wavelength:   {632.87} nm")
    print(f"Real Length:        {result['real_length_nm']:.3f} nm")
    print(f"Accepted c_j value: {result['c_difference_nm']} nm")
    print(f"Acceptance bounds:  {result['acceptance_interval']}")

except ValueError as e:
    print(f"Method Error: {e}")
except RuntimeError as e:
    print(f"Logic Error: {e}")

In [ ]:
wls = np.linspace(632, 633 ,100)
length_real = []

for wl in wls:
    he_ne_laser = MonochromaticEFMSolver(wavelength=wl)
    result = he_ne_laser.evaluate_block(
            nominal_length=100e6,           
            delta_nominal_length=-60.0,  # Tomado de certificado
            uncertainty=90.0,            # (0.03 + (6e-4)*100)*1e3 # nm | Tomado de certificado  
            measured_fraction=0.3908     # valor obtenido por el script para bloque de 100.0 mm
    )
    
    if result:
        length_real.append(result['real_length_nm'])
                
        print("Calibration Results:")
        print("-" * 20)
        print(f"Input wavelength:   {wl} nm")
        print(f"Real Length:        {result['real_length_nm']:.3f} nm")
        print(f"Accepted c_j value: {result['c_difference_nm']} nm")
        print(f"Acceptance bounds:  {result['acceptance_interval']}")
    else:
        length_real.append(result)
        print(f"Input wavelength:   {wl} nm")
        

    # except ValueError as e:
    #     length_real.append(0)
    #     print(f"Input wavelength:   {wl} nm")
    #     print(f"Method Error: {e}")
    # except RuntimeError as e:
    #     print(f"Logic Error: {e}")

In [ ]:
length_real

In [ ]:
plt.plot(wls[16:24], length_real[16:24])

In [ ]:
# Definición de Parámetros del Bloque Principal (100 mm, Acero)
L_nominal = 100.0                       # [mm]
l_fit_medido = 100.0000299499           # [mm] Obtenido por EFSW script propio
alpha_acero = 11.5e-6                   # [K^-1] (Coeficiente típico para bloques de acero)
bloque_100mm = GaugeBlockLengthCorrection(nominal_length=L_nominal, expansion_coefficient=alpha_acero)

# Datos para el Experimento de Paquete (Cambio de Fase)
# Bloques: 2mm, 5mm, 8mm, 10mm (n = 4)
num_blocks = 4
# Longitudes ópticas individuales (simuladas con un ligero desvío por rugosidad/fase)
l_opt_ind = [2.000015, 5.000010, 8.000012, 10.000018] 
suma_l_opt_ind = sum(l_opt_ind) # 25.000055 mm

# Forzamos que la diferencia nos de un l_phi de -0.000025 mm (-25 nm)
# l_phi = (l_opt_pack - suma_l_opt_ind) / 3  => l_opt_pack = suma_l_opt_ind + (3 * -0.000025)
l_opt_pack = 24.999980 # mm

In [ ]:
# Correcciones a la longitud calculada por MEFSW
